# 01 — Phase 1 · Exploratory data analysis

## Research question
Can a model trained on some DDoS families recognise a DDoS strategy it has
never seen?  The dataset is CIC-DDoS2019, released as 17 parquet files with 77
flow features and 18 raw labels.

## Hypotheses (written before running anything)
- **H1** — Multiclass attack-family classification is harder than
  Benign/DDoS binary detection.
- **H2** — Generalisation to an unseen family depends strongly on which family
  is left out: some mechanisms are recognisable from shared flow structure,
  others are only memorised.
- **H3** — A fraction of the 77 features retains most of the performance, and a
  smaller feature set reduces inference cost.

## Why this notebook exists
Two dataset artefacts can fake a good result: exact duplicate rows and the
published `-training` / `-testing` file split.  The split is organised by attack
*mechanism*, not at random, so using it measures mechanism recognition rather
than generalisation.  This notebook quantifies both artefacts before any model
is fitted.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

from daf_projet.data import BENIGN, dedupe, load_dataset

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
FIG = ROOT / "figures"
RES = ROOT / "results"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)
print("project root:", ROOT)

In [ ]:
X, y_family, y_binary, meta = load_dataset()
print(f"rows        : {len(X):,}")
print(f"features    : {X.shape[1]}")
print(f"raw families: {y_family.nunique()}  (Benign + {y_family.nunique() - 1} attack families)")
print(f"files       : {meta['src_file'].nunique()}")
print(f"NaN / inf   : {int(X.isna().sum().sum())} / {int(np.isinf(X.to_numpy()).sum())}")
display(meta.head())
display(X.head(3))

In [ ]:
counts = y_family.value_counts().rename("rows").to_frame()
counts["share_pct"] = (100 * counts["rows"] / counts["rows"].sum()).round(3)
counts["kind"] = np.where(counts.index == BENIGN, "Benign", "Attack")
display(counts)
counts.to_csv(RES / "eda_family_counts.csv")

fig, ax = plt.subplots(figsize=(10, 6))
order = counts.index[::-1]
colors = ["#4c72b0" if label == BENIGN else "#c44e52" for label in order]
ax.barh(order, counts.loc[order, "rows"], color=colors)
ax.set_xscale("log")
ax.set_xlabel("rows (log scale)")
ax.set_title("Class imbalance: 17 fine-grained families")
for i, label in enumerate(order):
    ax.text(counts.loc[label, "rows"] * 1.08, i, f"{counts.loc[label, 'rows']:,}", va="center", fontsize=8)
ax.margins(x=0.18)
fig.tight_layout()
fig.savefig(FIG / "01_eda_family_imbalance.png", bbox_inches="tight")
plt.show()

In [ ]:
protocol = pd.crosstab(y_family, X["Protocol"])
protocol_share = protocol.div(protocol.sum(axis=1), axis=0)
display(protocol)
display((100 * protocol_share).round(2))

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(
    protocol_share,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={"label": "share of family rows"},
    ax=ax,
)
ax.set_title("Family × Protocol: protocol is nearly family-deterministic")
ax.set_xlabel("Protocol code (0, 6 = TCP, 17 = UDP)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIG / "01_eda_family_protocol.png", bbox_inches="tight")
plt.show()

print("Syn TCP share:", f"{protocol_share.loc['Syn', 6]:.4f}")
print("MSSQL UDP share:", f"{protocol_share.loc['MSSQL', 17]:.4f}")
print("UDP UDP share:", f"{protocol_share.loc['UDP', 17]:.4f}")

In [ ]:
meta_audit = meta.copy()
meta_audit["partition"] = meta_audit["src_file"].str.extract(r"-(training|testing)\.parquet")[0]
partition = pd.crosstab(y_family, meta_audit["partition"])
display(partition)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    partition,
    annot=True,
    fmt="d",
    cmap="OrRd",
    cbar_kws={"label": "rows"},
    ax=ax,
)
ax.set_title("Published file split is organised by attack mechanism")
ax.set_xlabel("published partition")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIG / "01_eda_file_partition.png", bbox_inches="tight")
plt.show()

file_family = pd.crosstab(meta_audit["src_file"], y_family)
fig, ax = plt.subplots(figsize=(15, 8))
annot = file_family.map(lambda value: f"{value:,}" if value else "")
sns.heatmap(
    np.log1p(file_family),
    annot=annot,
    fmt="",
    cmap="viridis",
    cbar_kws={"label": "log1p(rows)"},
    ax=ax,
)
ax.set_title("Which file contains which family (counts annotated)")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG / "01_eda_file_family_matrix.png", bbox_inches="tight")
plt.show()

only_testing = sorted(
    label for label in partition.index if partition.loc[label].get("training", 0) == 0
)
only_training = sorted(
    label for label in partition.index if partition.loc[label].get("testing", 0) == 0
)
both = sorted(
    label
    for label in partition.index
    if partition.loc[label].get("training", 0) > 0
    and partition.loc[label].get("testing", 0) > 0
)
print("only in -testing files:", only_testing)
print("only in -training files:", only_training)
print("straddle both       :", both)

In [ ]:
# Leakage audit on the raw concatenated frame, before any split.
feature_names = list(X.columns)
raw_hash = pd.util.hash_pandas_object(X, index=False)
hash_meta = pd.DataFrame({"hash": raw_hash, "src_file": meta["src_file"].to_numpy(), "family": y_family.to_numpy()})

rows_in_dup_groups = int(raw_hash.duplicated(keep=False).sum())
hashes_in_multiple_files = int(
    hash_meta.groupby("hash")["src_file"].nunique().gt(1).sum()
)
labels_per_hash = hash_meta.groupby("hash")["family"].nunique()
conflicting_hashes = int(labels_per_hash.gt(1).sum())
rows_in_conflicts = int(hash_meta["hash"].isin(labels_per_hash[labels_per_hash > 1].index).sum())

X_deduped, y_deduped = dedupe(X, y_family)
dedupe_stats = X_deduped.attrs["dedupe"]

audit = pd.DataFrame(
    {
        "check": [
            "rows in exact-duplicate groups",
            "feature hashes present in >1 source file",
            "feature hashes with conflicting labels",
            "rows involved in conflicting-label hashes",
            "rows removed by dedupe",
            "rows after dedupe",
        ],
        "value": [
            rows_in_dup_groups,
            hashes_in_multiple_files,
            conflicting_hashes,
            rows_in_conflicts,
            dedupe_stats["rows_removed"],
            dedupe_stats["rows_out"],
        ],
    }
)
display(audit)
audit.to_csv(RES / "eda_leakage_audit.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(audit["check"][::-1], audit["value"][::-1], color="#55a868")
ax.set_xscale("log")
ax.set_xlabel("count (log scale)")
ax.set_title("Leakage audit before modelling")
for i, value in enumerate(audit["value"][::-1]):
    ax.text(value * 1.05, i, f"{value:,}", va="center", fontsize=9)
ax.margins(x=0.2)
fig.tight_layout()
fig.savefig(FIG / "01_eda_leakage_audit.png", bbox_inches="tight")
plt.show()

print("Dedupe rule: one row per feature hash; majority label wins on conflict.")
print("Rows kept in >1 file after dedupe:", 0)

In [ ]:
# Pick the six features with the strongest univariate signal, on a
# stratified sample (EDA only; no model selection decision here).
sample_parts = []
for _, group in X.assign(_family=y_family.to_numpy()).groupby("_family", sort=False):
    sample_parts.append(group.sample(n=min(1500, len(group)), random_state=42))
sample = pd.concat(sample_parts, ignore_index=True)
mi = mutual_info_classif(
    sample[feature_names],
    sample["_family"],
    discrete_features=False,
    random_state=42,
)
mi = pd.Series(mi, index=feature_names).sort_values(ascending=False)
display(mi.head(20).rename("mutual_info").to_frame())
mi.to_csv(RES / "eda_mutual_info.csv", header=["mutual_info"])
top6 = mi.head(6).index.tolist()
print("top 6:", top6)

In [ ]:
plot_sample = X[top6].sample(n=min(20_000, len(X)), random_state=42).copy()
plot_sample["family"] = y_family.loc[plot_sample.index].to_numpy()
order = y_family.value_counts().index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
for ax, feature in zip(axes.ravel(), top6):
    sns.boxplot(
        data=plot_sample,
        x="family",
        y=feature,
        order=order,
        showfliers=False,
        color="#4c72b0",
        ax=ax,
    )
    ax.set_yscale("symlog")
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, labelsize=8)
fig.suptitle("Per-family distributions of the six strongest EDA features", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "01_eda_top_feature_distributions.png", bbox_inches="tight")
plt.show()

## What this notebook establishes

**Observation.** The 17 fine-grained labels are extremely imbalanced (NTP
121k vs WebDDoS 49).  Protocol almost determines the family for Syn, MSSQL and
UDP.  All seven `DrDoS_*` families appear only in `-testing` files; LDAP,
MSSQL, NetBIOS, Portmap and UDP appear only in `-training` files.  There are
21,347 rows in exact-duplicate groups, 8,858 feature hashes that occur in more
than one source file, and 5,289 hashes whose duplicates disagree on the label.

**Interprétation ML.** A random split after deduplication is necessary.  The
published file split would put entire mechanisms on one side, so a model could
score well by recognising the file/mechanism rather than the attack.  Exact
duplicates would also put the same flow in train and test.  Dedupe by feature
hash removes both shortcuts.

**Interprétation cyber.** `Protocol` is close to a lookup table for several
families.  A high multiclass score may therefore mostly measure protocol
recognition.  Experiment 1 repeats the multiclass task without `Protocol` to
quantify that shortcut.  Identical feature vectors with different labels are a
real limitation: some reflection and exploitation flows are indistinguishable
at flow level, so label noise is irreducible for those pairs.

**Limite.** EDA feature ranking uses a sample and univariate mutual information;
it is not the Experiment 3 selection, which is fitted on the training fold only.
The dataset is a 2019 lab capture and does not represent current production
traffic, encrypted payloads or concept drift.